# Task 3: cek seed 43 dan 44 (Colab GPU)

8 run cls lr 2e-5 (~6 menit/run di L4, total ~50 menit): V1, V2, V3 profil baseline (`profiles_t2.parquet`), V3 profil JobBERT (`profiles_jobbert.parquet`), masing-masing untuk seed 43 dan 44. Seed mengubah split (per job) sekaligus seed training. Bobot tidak disimpan (hanya CSV dan prediksi), hasil lama di Drive tidak ditimpa: nama file berakhiran `_s43` / `_s44`.

Setelah selesai unduh `*_s43.*` dan `*_s44.*` dari `MyDrive/task3_match/results/` ke `data/results/`, lalu jalankan `python task3_match/seed_check.py`.

Jalankan semua cell berurutan.

In [2]:
BRANCH = "feat/task3-seed-check"
CKPT_ON_DRIVE = False

from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/task3_match"
CKPT = f"{OUT}/_ckpt" if CKPT_ON_DRIVE else "/content/ckpt"
!mkdir -p {OUT}/results {OUT}/models {OUT}/logs {CKPT}
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [3]:
import os, subprocess
tok = userdata.get("GITHUB_TOKEN")
REPO = "github.com/Ferdin-Arsenic/Job_Matching_System-NLP-.git"

def git(*args):
    """Jalankan git tanpa mencetak token: error disensor, dan remote disimpan tanpa token."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode: raise RuntimeError("git gagal: " + (r.stderr or r.stdout).replace(tok, "***"))
    return r.stdout

%cd /content
if not os.path.exists("/content/repo"):
    git("clone", "-q", "-b", BRANCH, f"https://x-access-token:{tok}@{REPO}", "repo")
    git("-C", "repo", "remote", "set-url", "origin", f"https://{REPO}")   # token tidak tersimpan di .git/config
else:
    git("-C", "repo", "pull", "-q", f"https://x-access-token:{tok}@{REPO}", BRANCH)
%cd /content/repo
print(git("log", "--oneline", "-1"))
!pip install -q datasets accelerate
!ls data/splits data/cache outputs/ner

/content
/content/repo
62de393 feat(task3): V3 with JobBERT profiles in analysis, --seed option, seed-check notebook

data/cache:
profiles_jobbert.metadata.json	profiles.parquet       sbert_minilm.npz
profiles_jobbert.parquet	profiles_stub.parquet  task2_metadata.json
profiles.metadata.json		profiles_t2.parquet    task2.parquet

data/splits:
task2.json  task3.json

outputs/ner:
ats_pairs.parquet    ner_ats_resume.parquet  t1_ner_results.csv
ner_atlas.parquet    README.md		     t1_seed_check.csv
ner_ats_job.parquet  t1_crf_check.csv	     t1_variant_comparison.csv


In [4]:
import glob, time, pandas as pd

def run(variant, profiles, seed, lr=2e-5):
    tag = f"{variant}{'-jbprof' if 'jobbert' in profiles else ''}_cls_lr{lr:g}_s{seed}"
    if os.path.exists(f"{OUT}/results/task3_ce_{tag}.csv"):
        print("skip (sudah ada di Drive):", tag); return
    t0 = time.time()
    cmd = (f"set -o pipefail; python task3_match/cross_encoder.py --variant {variant} --run cls --lr {lr} --seed {seed} "
           f"--profiles {profiles} --ckpt-dir {CKPT} --no-save 2>&1 | tee -a {OUT}/logs/{tag}.log")
    subprocess.run(["bash", "-c", cmd], check=True)
    os.system(f"cp data/results/task3_ce_{tag}.csv data/results/task3_pred_{tag}.parquet {OUT}/results/")
    print(f"selesai {tag} dalam {(time.time() - t0) / 60:.1f} menit")


In [5]:
for seed in (43, 44):
    for variant, profiles in (("V1", "profiles_stub.parquet"), ("V2", "profiles_stub.parquet"),
                              ("V3", "profiles_t2.parquet"), ("V3", "profiles_jobbert.parquet")):
        run(variant, profiles, seed)

selesai V1_cls_lr2e-05_s43 dalam 3.8 menit
selesai V2_cls_lr2e-05_s43 dalam 5.9 menit
selesai V3_cls_lr2e-05_s43 dalam 4.6 menit
selesai V3-jbprof_cls_lr2e-05_s43 dalam 3.4 menit
selesai V1_cls_lr2e-05_s44 dalam 5.9 menit
selesai V2_cls_lr2e-05_s44 dalam 4.6 menit
selesai V3_cls_lr2e-05_s44 dalam 5.8 menit
selesai V3-jbprof_cls_lr2e-05_s44 dalam 3.3 menit


In [6]:
r = pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUT}/results/task3_ce_*_s4[34].csv")])
print(r.query("split == 'test' and metric in ['macro_F1', 'ROC_AUC']").assign(profiles=lambda x: x.profiles.fillna(""))
       .pivot_table(index=["variant", "profiles"], columns=["metric"], values="value", aggfunc="first").round(3))

metric                            ROC_AUC  macro_F1
variant profiles                                   
V1                                  0.623     0.319
V2                                  0.684     0.441
V3      profiles_jobbert.parquet    0.655     0.370
        profiles_t2.parquet         0.683     0.450
